# Single-Trial Walkthrough: 1 → 2 → 3 Atoms

Walk through the full DiffSimuQ pipeline with **one PSR trial** (`n_sample=1`)
at increasing atom counts. For each case we show:

1. **Hamiltonian** — what we're differentiating
2. **Hardware path** — compile → TweezerMapper → ops, TransportLog, PulseLedger
3. **Verify** — per-segment `||H_compiled - H_target||` norm + gradient error
4. **Gradient check** — PSR sign matches finite difference

### Key architecture points

- **Ledger honesty**: dressing entries store the solver's H (from `o_coef + sol_gvars`), not the pre-compilation H
- **Solver channel selection**: single-qubit targets use detuning+rabi only — no dressing/ZZ
- **Kick compilation**: PSR kick (segment 1) maps Hj directly to channels, not through the solver's boxes
- **Verify norm diagnostics**: `||H_compiled - H_target||` per segment pinpoints compilation errors

---
## Setup

In [ ]:
import sys
sys.path.insert(0, "/Users/syue99/research/SimuQ/src/")
sys.path.insert(0, "/Users/syue99/research/SimuQ/differential_computing/")

# Force reload to pick up local changes
import importlib
# Clear all cached simuq + differential_computing submodules
to_remove = [k for k in sys.modules if k.startswith('simuq')
             or k in ('tweezer_mapper', 'pulse_ledger', 'aod_channel',
                       'verify_compilation', 'observable_program_generator',
                       'qutip_sequential', 'combine_gradient')]
for k in to_remove:
    del sys.modules[k]

import numpy as np
import sympy as sp
import qutip as qp

from simuq import QSystem, Qubit
from simuq.braket.diffQC_provider import diffQCProvider
from observable_program_generator import observable_program_generator
from qutip_sequential import QuTiPSequentialRunner
from combine_gradient import combine_gradient_results

# Verify solver has body-count filter
from simuq.solver import _body_count
print(f"Solver body-count filter: present")

T = 0.5
x_val = 1.0

def compile_provider(H_param, n_qubits):
    prov = diffQCProvider()
    qs_c = QSystem(); q_c = [Qubit(qs_c) for _ in range(n_qubits)]
    H_eval = H_param.set_parameterizedHam({"x": x_val})
    qs_c.add_evolution(H_eval, T)
    prov.compile(qs_c, "quera", "Aquila", "rydberg2d", tol=0.1, verbose=0)
    return prov

def single_trial(H_param, seed=42):
    np.random.seed(seed)
    return observable_program_generator(H_param, T, n_sample=1, n_repetition=1,
                                         diff_var="x", value=x_val)

def fd_gradient(H_param, n_qubits, obs, eps=1e-4):
    psi0 = qp.tensor([qp.basis(2, 0)] * n_qubits)
    def f(xv):
        He = H_param.set_parameterizedHam({"x": xv})
        r = qp.sesolve(He.to_qutip_qobj(), psi0, [0, T])
        return float(qp.expect(obs, r.states[-1]).real)
    return (f(x_val + eps) - f(x_val - eps)) / (2 * eps)

def show_ops(ops):
    for i, op in enumerate(ops):
        dur_ns = int(op["duration"] * 1000)
        if op["op"] == "aod":
            print(f"  [{i:2d}] AOD   amp={op['amplitude']:.3f}  dur={dur_ns} ns")
        elif op["op"] == "play":
            print(f"  [{i:2d}] PLAY  ch={op['channel']}  amp={op['amplitude']:.6f}  "
                  f"phase={op.get('phase', 0):.4f}  dur={dur_ns} ns")
        elif op["op"] == "delay":
            print(f"  [{i:2d}] DELAY dur={dur_ns} ns")

def show_play_entries(ledger):
    for e in ledger.play_entries():
        pos_str = ", ".join(f"({x:.1f},{y:.1f})" for x, y in e.positions)
        print(f"  [{e.step_idx:2d}] {e.channel_kind:10s} q={e.target_qubits}  "
              f"amp={e.amplitude}  dur={e.duration:.4f}us  "
              f"zones={e.zone}  H={'yes' if e.hamiltonian else 'no'}")

print("Setup complete.")

---
## Case 1: Single-qubit physics (2-atom register)

**Hamiltonian:** $H(x) = x \cdot Z_0 + X_0$

Only qubit 0 is parameterized; qubit 1 is idle. We use 2 atoms because
`rydberg2d` AAIS requires $\geq 2$ for its pairwise dressing sum.

**Solver channel selection**: body-count filter ensures only detuning + rabi
are used — no dressing, no ZZ. Observable: $\langle Z_0 \rangle$.

In [ ]:
# ── Build + compile + run ──
x = sp.Symbol("x")
qs1 = QSystem(); q1 = [Qubit(qs1) for _ in range(2)]
H1 = x * q1[0].Z + q1[0].X

prov1 = compile_provider(H1, n_qubits=2)
programs1 = single_trial(H1)
prov1.run(programs1, None, T=T, backend="hardware", verbose=1)

# Check: no dressing or ZZ in boxes
boxes = prov1.prog[2]
print("\nSolver boxes:")
for box_entries, dur in boxes:
    for (li, ji), ins, h_eval, lvars in box_entries:
        print(f"  {ins.name}: lvars={[f'{v:.4f}' for v in lvars]}")

In [ ]:
# ── Schedule ops + ledger ──
ops1 = prov1._branch_ops[0][0][0]
print(f"Branch 0: {len(ops1)} ops")
show_ops(ops1)

print("\n--- Play entries ---")
show_play_entries(prov1._pulse_ledgers[0][0])

# Confirm: no dressing or ZZ entries
kinds = {e.channel_kind for e in prov1._pulse_ledgers[0][0].play_entries()}
print(f"\nChannel kinds present: {kinds}")
assert 'dressing' not in kinds, "Should have no dressing for single-qubit H"
assert 'zz' not in kinds, "Should have no ZZ for single-qubit H"
print("No dressing or ZZ channels used (correct)")

In [ ]:
# ── Verify: norm diagnostics + gradient ──
obs1 = qp.tensor(qp.sigmaz(), qp.qeye(2))
result1 = prov1.verify(programs1, obs1, T=T,
                       psi0=QuTiPSequentialRunner(2).zero_state(), verbose=1)

# Gradient sign check
runner1 = QuTiPSequentialRunner(n_qubits=2)
expfn1 = runner1.make_expectation_fn(runner1.zero_state(), obs1)
grad_psr1 = combine_gradient_results(programs1, expfn1, T)
grad_fd1 = fd_gradient(H1, 2, obs1)
print(f"\nPSR gradient: {grad_psr1:.6f}")
print(f"FD  gradient: {grad_fd1:.6f}")
print(f"Sign match: {np.sign(grad_psr1) == np.sign(grad_fd1)}")

---
## Case 2: Two-qubit dressing interaction

**Hamiltonian:** $H(x) = \sin(2x) \cdot (Z_0 Z_1 + X_0 + X_1)$

The compiler maps $J \cdot Z_0 Z_1$ to dressing (global Rydberg potential).
Observable: $\langle Z_0 Z_1 \rangle$.

**Kick compilation**: PSR kicks include $Z_0 Z_1$ (needs gate-zone transport),
$X_0$, and $X_1$ (single-qubit, no transport). Each kick Hj is compiled
independently from the evolution segments.

In [ ]:
# ── Build + compile + run ──
x = sp.Symbol("x")
qs2 = QSystem(); q2 = [Qubit(qs2) for _ in range(2)]
J = sp.sin(2 * x)
H2 = J * q2[0].Z * q2[1].Z + J * q2[0].X + J * q2[1].X

prov2 = compile_provider(H2, n_qubits=2)
programs2 = single_trial(H2)
prov2.run(programs2, None, T=T, backend="hardware", verbose=1)

# Show PSR terms and their kick operators
print(f"\n{len(programs2)} PSR terms:")
for ti, (H_tot_list, ugrad, n_rep) in enumerate(programs2):
    Hj = H_tot_list[0][1][0]  # branch 0, seg 1, the H
    terms = [(t.to_tuple(), float(c)) for t, c in Hj.ham]
    print(f"  term {ti}: Hj = {terms}  ugrad = {ugrad:.4f}")

In [ ]:
# ── Schedule ops for ZZ kick branch (term 0) ──
ops2_zz = prov2._branch_ops[0][0][0]
print(f"Term 0 (Z0Z1 kick), branch 0: {len(ops2_zz)} ops")
show_ops(ops2_zz)

# Check: ZZ kick uses gate-zone transport
cz_moves = sum(len(log.cz_moves) for log in prov2._transport_logs[0])
print(f"\nCZ moves for term 0: {cz_moves} (gate-zone transport for ZZ kick)")

# Schedule ops for X kick branch (term 1)
ops2_x = prov2._branch_ops[1][0][0]
print(f"\nTerm 1 (X0 kick), branch 0: {len(ops2_x)} ops")
show_ops(ops2_x)

In [ ]:
# ── Verify ──
runner2 = QuTiPSequentialRunner(n_qubits=2)
obs2 = runner2.zz_observable(0, 1)
print("=== Verify ===")
result2 = prov2.verify(programs2, obs2, T=T, psi0=runner2.zero_state(), verbose=1)

# Gradient check
expfn2 = runner2.make_expectation_fn(runner2.zero_state(), obs2)
grad_psr2 = combine_gradient_results(programs2, expfn2, T)
grad_fd2 = fd_gradient(H2, 2, obs2)
print(f"\nPSR gradient: {grad_psr2:.6f}")
print(f"FD  gradient: {grad_fd2:.6f}")
print(f"Sign match: {np.sign(grad_psr2) == np.sign(grad_fd2)}")

---
## Case 3: Three-qubit system

**Hamiltonian:** $H(x) = \sin(2x) \cdot (Z_0 Z_1 + X_0 + X_1)$

Same coupling as Case 2, but with a 3rd spectator atom.
Observable: $\langle Z_0 Z_1 \rangle$.

In [ ]:
# ── Build + compile + run ──
x = sp.Symbol("x")
qs3 = QSystem(); q3 = [Qubit(qs3) for _ in range(3)]
J = sp.sin(2 * x)
H3 = J * q3[0].Z * q3[1].Z + J * q3[0].X + J * q3[1].X

prov3 = compile_provider(H3, n_qubits=3)
programs3 = single_trial(H3)
prov3.run(programs3, None, T=T, backend="hardware", verbose=1)

sol = prov3.prog[1]
print(f"\nsol_gvars = {sol}")
print(f"Atom positions: q0=(0,0), q1=({sol[0]:.3f},{sol[1]:.3f}), q2=({sol[2]:.3f},{sol[3]:.3f})")

In [ ]:
# ── Verify ──
runner3 = QuTiPSequentialRunner(n_qubits=3)
obs3 = runner3.zz_observable(0, 1)
print("=== Verify ===")
result3 = prov3.verify(programs3, obs3, T=T, psi0=runner3.zero_state(), verbose=1)

expfn3 = runner3.make_expectation_fn(runner3.zero_state(), obs3)
grad_psr3 = combine_gradient_results(programs3, expfn3, T)
grad_fd3 = fd_gradient(H3, 3, obs3)
print(f"\nPSR gradient: {grad_psr3:.6f}")
print(f"FD  gradient: {grad_fd3:.6f}")
print(f"Sign match: {np.sign(grad_psr3) == np.sign(grad_fd3)}")

---
## DSL Schedule Emission (2-atom, X-kick branch)

Emit the 2-atom X-kick branch ops through `to_pulsedsl_simple()`.
This branch has no AOD transport (single-qubit kick, atoms stay at interaction zone).

In [ ]:
import sys
sys.path.insert(0, "/Users/syue99/research/RISC-Q/PulseDSL/src/DSL/")

from PulseDSL_py import Channels, Schedule, PulseLib
from PulseDSL_py.pulselib import set_platform
from simuq.braket.diffQC_provider import to_pulsedsl_simple

n_channels = 8
ch, reg = Channels(n_channels)
schedule = Schedule()
set_platform(PulseLib.Rydberg)
aod_ch = ch[n_channels - 1]

# Use the X-kick branch (no AOD, cleaner schedule)
to_pulsedsl_simple(ops2_x, ch, aod_ch)
print(f"\nEmitted {len(ops2_x)} ops to PulseDSL schedule")

---
## Summary

### Verify results

| Case | Seg 0,2 norm | Seg 1 norm | Gradient error | Note |
|------|-------------|------------|----------------|------|
| 1q | ~2.8 | **0.00** | ~7% | Evolution segs have solver coeff mismatch |
| 2q | ~4.4 | **0.00** | ~55% | Dressing decomposition error |
| 3q | ~19 | **0.00** | ~15% | Larger system, more channels |

- **Seg 1 = 0 for all cases** — kick is compiled correctly (direct Hj mapping)
- **Seg 0,2 > 0** — solver coefficient mismatch (H_eff ≠ H_target per unit time)
- Gradient error is driven by evolution segment compilation quality

### Architecture

| Component | What it does |
|-----------|-------------|
| `_map_kick_segment` | Maps Hj directly: Z→detuning, X/Y→rabi, ZZ→gate zone + return |
| `_build_dressing_H` | Computes solver's dressing H from `o_coef + sol_gvars` |
| `_body_count` filter | Prevents dressing/ZZ from matching single-qubit targets |
| `norm_check` | Per-segment `\|\|H_compiled - H_target\|\|` for debugging |
| `to_pulsedsl_simple` | Emits sigmatukey pulses to PulseDSL channels |